# Session 5 — Flow Analysis & Queueing Theory for Simulation

**Course:** Decision Modelling — Simulation Part
**Session:** 5 of 12 (simulation portion)

## Learning objectives

By the end of this session you should be able to:

- Analyze a flow system using process-flow concepts and Little's Law
- Compute basic queueing KPIs (utilization, WIP, waiting time, cycle time) for M/M/1 and M/M/c systems, and approximate them for more general systems
- Prepare conceptually for building Discrete-Event Simulation models

## Where this session fits

- **Looking back:** Sessions 3–4 modeled "one-shot" decisions with Monte Carlo. Today we shift to systems that **evolve continuously over time** — queues, servers, flows — which is what Discrete-Event Simulation (Sessions 6–7) will let us build directly in code.
- **Today's role is deliberately analytical, not a coding-heavy session.** The formulas you derive and compute here are not just theory for its own sake: in Session 6, when you build your very first SimPy model (a single-server queue), you will compare its simulated output *against exactly these formulas* as a check that your model is behaving correctly. This is your first taste of **verification** — confirming a model computes what it's supposed to — which Session 12 treats formally.
- **Looking further ahead:** the cost-based server-sizing exercise at the end of this session is a small, self-contained preview of the kind of "search over a decision variable to optimize an objective" thinking that the integration/simulation-based-optimization sessions build on at scale.


## Further reading (supplementary, not required)

- "The Public Impact of Queueing Theory" — a paper on real-world applications and the societal relevance of queueing models.
- The Zara ("Zara's Secret for Fast Fashion") case is a good real-world illustration of flow/capacity thinking in a supply chain context, if you want an application beyond service systems.
- For the G/G/c approximation used below (the "VUT equation"), Hopp & Spearman's *Factory Physics* is the standard OSCM reference, if you want to go deeper than what's derived here.


## 1. Process flow analysis and Little's Law

Recall the flow vocabulary from Session 1: **WIP** (work in process, `L`), **throughput** (`TH`, the rate of completions), and **cycle time** (`CT` or `W`, time in the system). Session 1 only asked you to build intuition for these; today we make the relationship between them precise.

**Little's Law:**

```
L = TH x W
```

(Work-in-process = throughput x cycle time.)

What makes Little's Law remarkably useful is that it holds for **almost any stable queueing system**, regardless of the arrival process, service time distribution, number of servers, or queue discipline — as long as the system is in steady state (not systematically growing or shrinking). This generality is exactly why it will be your go-to sanity check on *any* DES model's output later in the course: no matter how complex the SimPy model, `L`, `TH`, and `W` measured from its output must satisfy Little's Law. If they don't, something in your model or your measurement is wrong.


## 2. The M/M/1 queue

The simplest queueing model: **exponential interarrival times** (Poisson arrivals), **exponential service times**, **1 server**, first-come-first-served, infinite waiting room. Despite its simplicity, it's the natural starting point for verifying a single-server SimPy model in Session 6.

Let `lambda` = arrival rate, `mu` = service rate, and `rho = lambda / mu` (utilization). For a stable system we need `rho < 1`. Standard results:

```
P0 (probability system is empty) = 1 - rho
L  (avg number in system)          = rho / (1 - rho)
Lq (avg number in queue)           = rho^2 / (1 - rho)
W  (avg time in system)            = 1 / (mu - lambda)
Wq (avg time in queue)             = rho / (mu - lambda)
```

Note `L = lambda * W` and `Lq = lambda * Wq` — Little's Law applied to the whole system and to the queue alone, respectively.


In [1]:
def mm1(lam, mu):
    rho = lam / mu
    if rho >= 1:
        raise ValueError(f"Unstable system: rho = {rho:.3f} >= 1")
    P0 = 1 - rho
    L = rho / (1 - rho)
    Lq = rho**2 / (1 - rho)
    W = 1 / (mu - lam)
    Wq = rho / (mu - lam)
    return {'rho': rho, 'P0': P0, 'L': L, 'Lq': Lq, 'W': W, 'Wq': Wq}

# Sanity check on Little's Law consistency
result = mm1(lam=0.3, mu=0.5)
print(result)
print(f"Check: lambda * W = {0.3 * result['W']:.4f}  (should equal L = {result['L']:.4f})")


{'rho': 0.6, 'P0': 0.4, 'L': 1.4999999999999998, 'Lq': 0.8999999999999999, 'W': 5.0, 'Wq': 2.9999999999999996}
Check: lambda * W = 1.5000  (should equal L = 1.5000)


## 3. Worked example: the airport security checkpoint (M/M/1)

Consider passenger arrivals at an airport security checkpoint. Each passenger is a job; the X-ray scanner is the server, FCFS order. We observed 9 interarrival times and 10 processing times (in seconds):


In [2]:
import numpy as np

interarrival_times = np.array([10, 10, 2, 10, 1, 3, 7, 9, 2])
processing_times = np.array([7, 1, 7, 2, 8, 7, 4, 8, 5, 1])

mean_interarrival = interarrival_times.mean()
mean_processing = processing_times.mean()

lam = 1 / mean_interarrival   # arrival rate, per second
mu = 1 / mean_processing      # service rate, per second

print(f"Mean interarrival time: {mean_interarrival:.3f} s  ->  lambda = {lam:.4f} per second")
print(f"Mean processing time:   {mean_processing:.3f} s  ->  mu = {mu:.4f} per second")

result_mm1 = mm1(lam, mu)
for k, v in result_mm1.items():
    print(f"{k}: {v:.4f}")


Mean interarrival time: 6.000 s  ->  lambda = 0.1667 per second
Mean processing time:   5.000 s  ->  mu = 0.2000 per second
rho: 0.8333
P0: 0.1667
L: 5.0000
Lq: 4.1667
W: 30.0000
Wq: 25.0000


This M/M/1 analysis **assumes** exponential interarrival and service times — a strong assumption for a sample this small. What if we don't want to assume a specific distribution shape, just work with the sample mean and variability we actually observed? That's exactly what the G/G/1 approximation below is for.


## 4. Beyond exponential: the G/G/1 approximation (the "VUT equation")

When interarrival and service times follow **general** (not necessarily exponential) distributions, there's no simple exact formula for `Wq` — but a well-known and widely used **approximation** exists, sometimes called the VUT equation (Variability-Utilization-Time):

```
Wq(G/G/1)  ~=  ( (ca^2 + cs^2) / 2 )  x  ( rho / (1 - rho) )  x  ts
```

where `ca^2` and `cs^2` are the **squared coefficients of variation** (variance divided by mean-squared) of the interarrival times and service times respectively, `rho = lambda x ts` is utilization, and `ts` is the mean service time. Intuitively: waiting time scales with *variability* (V), *utilization* (U), and the *time* scale of service (T) — hence "VUT." Note that when `ca^2 = cs^2 = 1` (the exponential case, since the exponential distribution has SCV = 1), this reduces exactly to the M/M/1 formula for `Wq`.


In [3]:
def scv(data):
    # Squared coefficient of variation: (std/mean)^2, using sample statistics.
    data = np.asarray(data, dtype=float)
    mean = data.mean()
    std = data.std(ddof=1)
    return (std / mean) ** 2

def gg1_wq(lam, mean_service, ca2, cs2):
    rho = lam * mean_service
    if rho >= 1:
        raise ValueError(f"Unstable system: rho = {rho:.3f} >= 1")
    return ((ca2 + cs2) / 2) * (rho / (1 - rho)) * mean_service

ca2 = scv(interarrival_times)
cs2 = scv(processing_times)
print(f"ca^2 (interarrival SCV) = {ca2:.4f}")
print(f"cs^2 (processing SCV)   = {cs2:.4f}")

wq_gg1 = gg1_wq(lam, mean_processing, ca2, cs2)
print(f"\nWq (G/G/1 approximation) = {wq_gg1:.4f} seconds")
print(f"Wq (M/M/1 exact, for comparison) = {result_mm1['Wq']:.4f} seconds")


ca^2 (interarrival SCV) = 0.4306
cs^2 (processing SCV)   = 0.3200

Wq (G/G/1 approximation) = 9.3819 seconds
Wq (M/M/1 exact, for comparison) = 25.0000 seconds


### Discussion

The G/G/1 approximation and the M/M/1 exact result will generally differ, since the small sample's SCVs aren't exactly 1 (the exponential's SCV). Which do you trust more here — the exact M/M/1 formula under an assumption the data may not fully support, or the approximation using the sample's actual variability, given how small the sample is? (There's no single right answer — this is exactly the kind of judgment call input analysis, Session 2, is meant to help you make on real data.)


## 5. Multi-server queues: M/M/c

Now suppose there are `c` identical servers sharing one queue (e.g. multiple X-ray scanners, multiple doctors). Let `a = lambda / mu` (the *offered load*) and `rho = a / c` (per-server utilization, needs `rho < 1` for stability).

```
P0 = [ sum_{n=0}^{c-1} (a^n / n!)  +  (a^c / (c! (1-rho))) ]^(-1)

P_wait = (a^c / (c! (1-rho))) x P0          (Erlang C: probability an arrival must wait)

Lq = P_wait x rho / (1 - rho)
Wq = Lq / lambda
W  = Wq + 1/mu
L  = lambda x W
```


In [4]:
from math import factorial

def mmc(lam, mu, c):
    a = lam / mu           # offered load
    rho = a / c            # per-server utilization
    if rho >= 1:
        raise ValueError(f"Unstable system: rho = {rho:.3f} >= 1")

    sum_terms = sum((a**n) / factorial(n) for n in range(c))
    last_term = (a**c) / (factorial(c) * (1 - rho))
    P0 = 1 / (sum_terms + last_term)

    P_wait = last_term * P0
    Lq = P_wait * rho / (1 - rho)
    Wq = Lq / lam
    W = Wq + 1 / mu
    L = lam * W

    return {'rho': rho, 'P0': P0, 'P_wait': P_wait, 'L': L, 'Lq': Lq, 'W': W, 'Wq': Wq}

# Quick internal consistency check: c=1 should reduce to the M/M/1 formulas
check = mmc(lam=0.3, mu=0.5, c=1)
reference = mm1(lam=0.3, mu=0.5)
print("M/M/c with c=1:", {k: round(v, 4) for k, v in check.items() if k in reference})
print("M/M/1 direct:   ", {k: round(v, 4) for k, v in reference.items()})


M/M/c with c=1: {'rho': 0.6, 'P0': 0.4, 'L': 1.5, 'Lq': 0.9, 'W': 5.0, 'Wq': 3.0}
M/M/1 direct:    {'rho': 0.6, 'P0': 0.4, 'L': 1.5, 'Lq': 0.9, 'W': 5.0, 'Wq': 3.0}


### Sanity check against real, previously-computed results

Here's a genuine test of our implementation: a doctor's office model from a prior course iteration includes a table of **pre-computed M/M/6 results** for several `(lambda, mu)` pairs, computed independently in Excel. Let's check our Python implementation reproduces one of those rows exactly.

**Given (from the Excel table):** `lambda = 1.2`, `mu = 0.5`, `c = 6` servers, with reported results `L = 2.4266`, `Lq = 0.026635`, `W = 2.0222`, `Wq = 0.022196`, `P0 = 0.090315`.


In [5]:
check_mmc = mmc(lam=1.2, mu=0.5, c=6)

reference_values = {'L': 2.4266350683506253, 'Lq': 0.026635068350625703,
                     'W': 2.022195890292188, 'Wq': 0.022195890292188086,
                     'P0': 0.09031530880610388}

print(f"{'Metric':<6}{'Our function':>15}{'Reference (Excel)':>20}")
for k, ref_v in reference_values.items():
    our_v = check_mmc[k]
    print(f"{k:<6}{our_v:>15.6f}{ref_v:>20.6f}")


Metric   Our function   Reference (Excel)
L            2.426635            2.426635
Lq           0.026635            0.026635
W            2.022196            2.022196
Wq           0.022196            0.022196
P0           0.090315            0.090315


If these match closely, we now have a **verified** M/M/c implementation — exactly the kind of independent cross-check (comparing new code against a trusted prior result) that verification/validation practice relies on.


## 6. Pooled vs. dedicated capacity: a classic comparison

Back to the airport: suppose interarrival times are exponential with mean 6 seconds, and scanning times are exponential with mean 5 seconds. Management is adding a second X-ray machine and considers two scenarios:

- **Scenario 1 (pooled):** both scanners share one common queue — an M/M/2 system
- **Scenario 2 (dedicated):** each passenger is randomly sent (50/50) to one of two independent single-server queues — two separate M/M/1 systems, each seeing half the arrival rate


In [6]:
lam_airport = 1 / 6   # per second
mu_airport = 1 / 5    # per second

pooled = mmc(lam_airport, mu_airport, c=2)
dedicated = mm1(lam_airport / 2, mu_airport)

print("Scenario 1 (pooled, M/M/2):")
for k, v in pooled.items():
    print(f"  {k}: {v:.4f}")

print("\nScenario 2 (dedicated, M/M/1 x 2, each seeing half the arrivals):")
for k, v in dedicated.items():
    print(f"  {k}: {v:.4f}")

print(f"\nWq -- pooled:    {pooled['Wq']:.4f} seconds")
print(f"Wq -- dedicated: {dedicated['Wq']:.4f} seconds")


Scenario 1 (pooled, M/M/2):
  rho: 0.4167
  P0: 0.4118
  P_wait: 0.2451
  L: 1.0084
  Lq: 0.1751
  W: 6.0504
  Wq: 1.0504

Scenario 2 (dedicated, M/M/1 x 2, each seeing half the arrivals):
  rho: 0.4167
  P0: 0.5833
  L: 0.7143
  Lq: 0.2976
  W: 8.5714
  Wq: 3.5714

Wq -- pooled:    1.0504 seconds
Wq -- dedicated: 3.5714 seconds


This is one of the most robust results in queueing theory: **pooling capacity reduces waiting time**, even though the total service capacity is identical in both scenarios. Intuitively, dedicated queues can leave one server idle while the other has a backlog; pooling lets any idle server pick up any waiting customer. Keep this result in mind — it's a recurring theme in capacity-planning decisions throughout OSCM.


## Exercise (guided): finding the cost-optimal number of servers

Airport management incurs a **holding cost of $1 per unit of WIP per second** (`L`, from the pooled M/M/c model) and a cost of **$1 per server**. They want to know: what number of servers `c` minimizes total cost per second, `Total Cost(c) = holding_cost x L(c) + server_cost x c`?

Your task:

1. Loop over candidate values of `c` (try `c` from 1 to 8 — remember `c` must be large enough for stability, i.e. `rho = lambda/(c*mu) < 1`)
2. For each feasible `c`, compute `L` using `mmc()` and the total cost
3. Identify the cost-minimizing `c`
4. Plot total cost vs. `c`

This is a small preview of a pattern you'll see again, at much larger scale, in the simulation-based-optimization integration sessions later in the course: **evaluate an objective across candidate decisions and search for the best one.**


In [7]:
# EXERCISE
import matplotlib.pyplot as plt

holding_cost = 1.0
server_cost = 1.0
c_values = range(1, 9)

# TODO: for each c in c_values, check stability (rho < 1), compute L via mmc(),
#       and compute total_cost = holding_cost * L + server_cost * c
#       (skip / omit c values that are not stable)

# TODO: store results in a list of dicts or a DataFrame with columns c, L, total_cost

# TODO: identify the c with the minimum total_cost

# TODO: plot total_cost vs c


<details>
<summary>Solution (click to expand)</summary>

```python
# SOLUTION
import pandas as pd

rows = []
for c in c_values:
    rho = lam_airport / (c * mu_airport)
    if rho >= 1:
        continue
    metrics = mmc(lam_airport, mu_airport, c)
    total_cost = holding_cost * metrics['L'] + server_cost * c
    rows.append({'c': c, 'L': metrics['L'], 'total_cost': total_cost})

cost_df = pd.DataFrame(rows)
print(cost_df.round(4))

best_row = cost_df.loc[cost_df['total_cost'].idxmin()]
print(f"\nCost-minimizing number of servers: c = {best_row['c']:.0f}, "
      f"total cost = {best_row['total_cost']:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(cost_df['c'], cost_df['total_cost'], 'o-')
plt.xlabel('Number of servers (c)')
plt.ylabel('Total cost per second')
plt.title('Total cost vs. number of servers')
plt.show()
```
</details>


### Discussion questions

1. Why does adding *too many* servers eventually stop helping total cost, even though it always reduces `L`? What's driving the trade-off?
2. In Section 6 we saw pooling reduces waiting time. Does that same pooling logic apply to how you'd think about scaling `c` up in this cost exercise?
3. Every formula in this session assumed a **steady state**. What do you think "steady state" requires in practice (e.g. about how long the system has been running, or whether the arrival rate changes over the day)? Keep this question in mind — Session 9 (steady-state output analysis) addresses it directly for simulated systems.


## Wrap-up

Today we:

- Made Little's Law precise (`L = TH x W`) and established why its generality makes it a universal sanity check for simulation output
- Derived and implemented exact M/M/1 and M/M/c queueing formulas
- Learned the G/G/1 "VUT" approximation for when exponential assumptions don't hold
- **Verified** our M/M/c implementation against independently-computed results from a real prior model
- Demonstrated the classic pooled-vs-dedicated capacity result
- Used a queueing-cost trade-off as a first, small preview of decision-search thinking

**Next session:** Introduction to Discrete-Event Simulation (SimPy I) — we build our first SimPy model of a single-server queue, and use exactly the M/M/1 formulas from today to check that the simulation's output matches theory.
